## DATA PREPROCESSING

- word lengthening normalization (melakukan normalisasi kata pada tweet agar tidak terdapat pengulangan huruf secara berlebihan.
- slang normalization (mengubah kata-kata slang di indonesia termasuk singkatan, yang mana ini menggunakan kamus yang telah dibuat oleh orang lain, ada 15.396 kata pada kamusnya)
- tokenization  (membuat token-token berdasarkan kata)
- stemming  (mengubah teks agar tersusun oleh kata baku)

In [18]:
# 1.1 Import library utama
import pandas as pd
import re
import os

In [ ]:
# 1.2 Load dataset hasil data cleaning
df = pd.read_csv('../datacleaningcopy/data_cleaning_final.csv')

print("Dataset berhasil dimuat.")
print("Jumlah data:", len(df))
df.head()

Dataset berhasil dimuat.
Jumlah data: 13192


,no,timestamp,teks
0,1,2016-12-30T06:37:56.000Z,ADIL loh utk yg punya kebijakan publik negara ...
1,2,2016-12-30T06:30:36.000Z,Tertibkan Media Online DPR Pemerintah Jangan S...
2,3,2016-12-30T04:48:35.000Z,harus dievaluasi lg kebijakan bebas visa truta...
3,4,2016-12-30T04:21:40.000Z,jangan ngambang aturan logis apa undang undang
4,5,2016-12-30T02:36:13.000Z,Kebebasan bersuara berpendapat memang dijamin ...


In [1]:
import pandas as pd

pd.set_option('display.max_colwidth', None)

df = pd.read_csv('../datapreprocessingcopy/data_preprocessing_final.csv')
df.head()

,no,timestamp,teks,teks_processed
0,1,2016-12-30T06:37:56.000Z,ADIL loh utk yg punya kebijakan publik negara Ingat yg ini!!!,ADIL loh untuk yang punya kebijakan publik negara ingat yang ini ! !
1,2,2016-12-30T06:30:36.000Z,Tertibkan Media Online DPR Pemerintah Jangan Sporadis Apalagi Selektif Hanya kepada Media yang Berseberangan,tertib media online DPR pemerintah jangan sporadis apalagi selektif hanya kepada media yang bersebrangan
2,3,2016-12-30T04:48:35.000Z,harus dievaluasi lg kebijakan bebas visa trutama utk negara tiongkok pak!! bahaya tersembunyi martabat RI,harus evaluasi lagi kebijakan bebas visa utama untuk negara tiongkok pak ! ! bahaya sembunyi martabat RI
3,4,2016-12-30T04:21:40.000Z,jangan ngambang aturan logis apa undang undang,jangan ngambang pengaturan logis apa undang undang
4,5,2016-12-30T02:36:13.000Z,Kebebasan bersuara berpendapat memang dijamin UU Namun kebebasan tersebut tdk harus kebablasan sehingga menabrak aturan aturan logis,bebas suara dapat memang jamin UU tetapi bebas sebut tidak harus bablas sehingga tabrak pengaturan pengaturan logis


## WORD LENGTHENING NORMALIZATION

In [7]:
# 2.1 Fungsi untuk normalisasi word lengthening
def normalize_lengthening(text):
    if not isinstance(text, str):
        return text
    
    # Mengubah huruf berulang lebih dari 2 menjadi maksimal 2
    # contoh: bagussss → baguss, jelekkkk → jelekk
    text = re.sub(r'(.)\1{2,}', r'\1\1', text)
    
    return text

In [8]:
# 2.2 Menerapkan word lengthening pada kolom teks
df['teks_lengthening'] = df['teks'].apply(normalize_lengthening)

df[['teks', 'teks_lengthening']].head()

,teks,teks_lengthening
0,ADIL loh utk yg punya kebijakan publik negara ...,ADIL loh utk yg punya kebijakan publik negara ...
1,Tertibkan Media Online DPR Pemerintah Jangan S...,Tertibkan Media Online DPR Pemerintah Jangan S...
2,harus dievaluasi lg kebijakan bebas visa truta...,harus dievaluasi lg kebijakan bebas visa truta...
3,jangan ngambang aturan logis apa undang undang,jangan ngambang aturan logis apa undang undang
4,Kebebasan bersuara berpendapat memang dijamin ...,Kebebasan bersuara berpendapat memang dijamin ...


In [5]:
# 2.3 Cek perubahan data
jumlah_berubah = (df['teks'] != df['teks_lengthening']).sum()

print("Jumlah data yang mengalami perubahan:", jumlah_berubah)

Jumlah data yang mengalami perubahan: 849


In [6]:
# 2.4 Simpan hasil setelah word lengthening
os.makedirs('datapreprocessingcopy', exist_ok=True)

df.to_csv(
    os.path.join('datapreprocessingcopy', 'data_after_word_lengthening.csv'),
    index=False
)

print("data_after_word_lengthening.csv berhasil disimpan.")

data_after_word_lengthening.csv berhasil disimpan.


## Slang Normalization

In [7]:
# 2.1 Konfigurasi path
import os

BASE_DIR = 'D:/TA/ProyekScraping/PROJECT'
KAMUS_PATH = os.path.join(BASE_DIR, 'kamus', 'colloquial_lexicon.csv')
INPUT_PATH = os.path.join(BASE_DIR, 'datapreprocessingcopy', 'data_after_word_lengthening.csv')
OUTPUT_DIR = os.path.join(BASE_DIR, 'datapreprocessingcopy')

In [8]:
# 2.2 Load Kamus
import pandas as pd
import json

# Load kamus alay utama (slang → colloquial)
df_kamus = pd.read_csv(KAMUS_PATH)
slang_dict = dict(zip(df_kamus['slang'].astype(str), df_kamus['formal'].astype(str)))

# Tambah mapping: colloquial → formal KBBI (untuk kompatibilitas InSet)
colloquial_to_formal = {
    # === NEGASI ===
    'enggak': 'tidak',
    'nggak': 'tidak',
    'gak': 'tidak',
    'ga': 'tidak',
    'gk': 'tidak',
    'tdk': 'tidak',
    'bkn': 'bukan',
    'bukannya': 'bukan',
    
    # === INTENSIFIER / DEREGEE MODIFIERS ===
    'bgt': 'banget',
    'bgtd': 'banget',
    'amat': 'banget',
    'parah': 'banget',
    'gila': 'banget',
    'benar': 'banget',
    'betul': 'banget',
    
    # === CONJUNCTION ===
    'tapi': 'tetapi',
    'tp': 'tetapi',
    'namun': 'tetapi',
    'kalo': 'jika',
    'kalau': 'jika',
    'kl': 'jika',
    'klu': 'jika',
    
    # === COMMON SLANG → FORMAL ===
    'aja': 'saja',
    'udah': 'sudah',
    'udh': 'sudah',
    'dah': 'sudah',
    'sdh': 'sudah',
    'blm': 'belum',
    'lg': 'lagi',
    'yg': 'yang',
    'y': 'yang',
    'dgn': 'dengan',
    'dg': 'dengan',
    'krn': 'karena',
    'sebab': 'karena',
    'bs': 'bisa',
    'bisa': 'dapat',
    'kyk': 'seperti',
    'kayak': 'seperti',
    'gini': 'begini',
    'gitu': 'begitu',
    'nih': 'ini',
    'tu': 'itu',
    'itu': 'itu',
    'ini': 'ini',
    'sih': 'sih',  # partikel
    'lah': 'lah',  # partikel
    'kah': 'kah',  # partikel
    'dong': 'dong',  # partikel
    'deh': 'deh',  # partikel
    
    # === PRONOUNS ===
    'aq': 'aku',
    'gw': 'saya',
    'gue': 'saya',
    'gua': 'saya',
    'elu': 'kamu',
    'lu': 'kamu',
    'km': 'kamu',
    
    # === TIME/PLACE ===
    'skrg': 'sekarang',
    'ntar': 'nanti',
    'besok': 'besok',
    'kemaren': 'kemarin',
    'sini': 'sini',
    'situ': 'situ',
    'sana': 'sana',
}

# Merge: slang → formal (two-step mapping)
def two_step_normalize(word, slang_dict, formal_dict):
    """
    Two-step normalization:
    1. slang → colloquial (kamus-alay)
    2. colloquial → formal (custom mapping)
    """
    word_lower = word.lower()
    
    # Step 1: slang → colloquial
    colloquial = slang_dict.get(word_lower, word_lower)
    
    # Step 2: colloquial → formal
    formal = formal_dict.get(colloquial, colloquial)
    
    return formal

print(f"Total slang entries: {len(slang_dict)}")
print(f"Total colloquial→formal mappings: {len(colloquial_to_formal)}")
print(f"\nContoh two-step mapping:")
test_words = ['ga', 'nggak', 'bgt', 'tapi', 'aja']
for w in test_words:
    result = two_step_normalize(w, slang_dict, colloquial_to_formal)
    print(f"  {w} → {result}")

Total slang entries: 4331
Total colloquial→formal mappings: 64

Contoh two-step mapping:
  ga → tidak
  nggak → tidak
  bgt → banget
  tapi → tetapi
  aja → saja


In [9]:
# 2.3 Load Data Hasil Word Lengthening
df = pd.read_csv(INPUT_PATH)
print(f"Dataset berhasil dimuat: {len(df)} baris.")

Dataset berhasil dimuat: 13192 baris.


In [10]:
# 2.4 Fungsi Normalisasi Slang (Two-Step + Preservasi Kapitalisasi)
def normalize_slang(text, slang_dict, formal_dict):
    """
    Normalisasi slang dengan two-step mapping dan preservasi kapitalisasi.
    """
    if not isinstance(text, str):
        return text
    
    words = text.split()
    normalized_words = []
    
    for word in words:
        # Two-step normalization
        normalized = two_step_normalize(word, slang_dict, formal_dict)
        
        # Preservasi format kapitalisasi (khusus ALL CAPS untuk VADER)
        if word.isupper() and len(word) > 1:
            normalized = normalized.upper()
        elif word.istitle():
            normalized = normalized.capitalize()
        # else: biarkan lowercase
        
        normalized_words.append(normalized)
    
    return ' '.join(normalized_words)

In [11]:
# 2.5 Menerapkan Normalisasi (dengan two-step mapping)
df['teks_normalized'] = df['teks_lengthening'].apply(
    lambda x: normalize_slang(x, slang_dict, colloquial_to_formal)
)

print("\nSampel Perubahan (Two-Step):")
print(df[['teks_lengthening', 'teks_normalized']].head(10))


Sampel Perubahan (Two-Step):
                                    teks_lengthening  \
0  ADIL loh utk yg punya kebijakan publik negara ...   
1  Tertibkan Media Online DPR Pemerintah Jangan S...   
2  harus dievaluasi lg kebijakan bebas visa truta...   
3     jangan ngambang aturan logis apa undang undang   
4  Kebebasan bersuara berpendapat memang dijamin ...   
5  Anggota Komisi IX DPR Okky Asokawati Evaluasi ...   
6  Anggota DPR RI Komisi II Dari Fraksi NasDem Ge...   
7  Wapres Minta Kebijakan Bebas Visa Dievaluasi c...   
8  Asal Sesuai Undang Undang Anggota DPR Ini Setu...   
9  Saat memberikan makanan bergizi bantuan Kemenk...   

                                     teks_normalized  
0  ADIL loh untuk yang punya kebijakan publik neg...  
1  Tertibkan Media Online DPR Pemerintah Jangan S...  
2  harus dievaluasi lagi kebijakan bebas visa ter...  
3     jangan ngambang aturan logis apa undang undang  
4  Kebebasan bersuara berpendapat memang dijamin ...  
5  Anggota Komisi IX DP

In [ ]:
# 2.6 Simpan hasil setelah slang normalization
os.makedirs('datapreprocessingcopy', exist_ok=True)

df.to_csv(
    os.path.join('datapreprocessingcopy', 'data_after_slang_normalization.csv'),
    index=False
)

print("data_after_slang_normalization.csv berhasil disimpan.")

data_after_slang_normalization.csv berhasil disimpan.


In [13]:
# 2.7 Test fungsi preservasi kapitalisasi + two-step mapping
test_cases = [
    "utk apa YG bgtt",
    "UTK APA YG BGTT",
    "Utk Apa Yg Bgtt",
    "DPR harus evaluasi LG kebijakan ini",
    "ga enak tapi bgt sih",
    "GAK SUKA TAPI BAGUS"
]

print("Test Two-Step Normalization + Preservasi Kapitalisasi:\n")
for test in test_cases:
    result = normalize_slang(test, slang_dict, colloquial_to_formal)
    print(f"Original:   {test}")
    print(f"Normalized: {result}")
    print("-" * 60)

Test Two-Step Normalization + Preservasi Kapitalisasi:

Original:   utk apa YG bgtt
Normalized: untuk apa YANG banget
------------------------------------------------------------
Original:   UTK APA YG BGTT
Normalized: UNTUK APA YANG BANGET
------------------------------------------------------------
Original:   Utk Apa Yg Bgtt
Normalized: Untuk Apa Yang Banget
------------------------------------------------------------
Original:   DPR harus evaluasi LG kebijakan ini
Normalized: DPR harus evaluasi LAGI kebijakan ini
------------------------------------------------------------
Original:   ga enak tapi bgt sih
Normalized: tidak enak tetapi banget sih
------------------------------------------------------------
Original:   GAK SUKA TAPI BAGUS
Normalized: TIDAK SUKA TETAPI BAGUS
------------------------------------------------------------


## Tokenization

In [14]:
# 3.1 Load data hasil slang normalization
import pandas as pd

df = pd.read_csv('datapreprocessingcopy/data_after_slang_normalization.csv')

print("Dataset berhasil dimuat.")
print("Jumlah data:", len(df))
df.head()

Dataset berhasil dimuat.
Jumlah data: 13192


,no,timestamp,teks,teks_lengthening,teks_normalized
0,1,2016-12-30T06:37:56.000Z,ADIL loh utk yg punya kebijakan publik negara ...,ADIL loh utk yg punya kebijakan publik negara ...,ADIL loh untuk yang punya kebijakan publik neg...
1,2,2016-12-30T06:30:36.000Z,Tertibkan Media Online DPR Pemerintah Jangan S...,Tertibkan Media Online DPR Pemerintah Jangan S...,Tertibkan Media Online DPR Pemerintah Jangan S...
2,3,2016-12-30T04:48:35.000Z,harus dievaluasi lg kebijakan bebas visa truta...,harus dievaluasi lg kebijakan bebas visa truta...,harus dievaluasi lagi kebijakan bebas visa ter...
3,4,2016-12-30T04:21:40.000Z,jangan ngambang aturan logis apa undang undang,jangan ngambang aturan logis apa undang undang,jangan ngambang aturan logis apa undang undang
4,5,2016-12-30T02:36:13.000Z,Kebebasan bersuara berpendapat memang dijamin ...,Kebebasan bersuara berpendapat memang dijamin ...,Kebebasan bersuara berpendapat memang dijamin ...


In [15]:
# 3.2 Fungsi Tokenization
import re

def tokenize_text(text):
    if not isinstance(text, str):
        return []
    
    # Pisahkan kata dan tanda baca
    tokens = re.findall(r'\w+|[^\w\s]', text, flags=re.UNICODE)
    
    # Hapus token kosong
    tokens = [t for t in tokens if t.strip()]
    
    return tokens

In [16]:
# 3.3 Menerapkan tokenization
df['tokens'] = df['teks_normalized'].apply(tokenize_text)

df[['teks_normalized', 'tokens']].head()

,teks_normalized,tokens
0,ADIL loh untuk yang punya kebijakan publik neg...,"[ADIL, loh, untuk, yang, punya, kebijakan, pub..."
1,Tertibkan Media Online DPR Pemerintah Jangan S...,"[Tertibkan, Media, Online, DPR, Pemerintah, Ja..."
2,harus dievaluasi lagi kebijakan bebas visa ter...,"[harus, dievaluasi, lagi, kebijakan, bebas, vi..."
3,jangan ngambang aturan logis apa undang undang,"[jangan, ngambang, aturan, logis, apa, undang,..."
4,Kebebasan bersuara berpendapat memang dijamin ...,"[Kebebasan, bersuara, berpendapat, memang, dij..."


In [17]:
# 3.4 Cek hasil tokenization
print("Contoh token:")
for i in range(5):
    print(df['tokens'].iloc[i])

Contoh token:
['ADIL', 'loh', 'untuk', 'yang', 'punya', 'kebijakan', 'publik', 'negara', 'Ingat', 'yang', 'ini', '!', '!']
['Tertibkan', 'Media', 'Online', 'DPR', 'Pemerintah', 'Jangan', 'Sporadis', 'Apalagi', 'Selektif', 'Hanya', 'kepada', 'Media', 'yang', 'Berseberangan']
['harus', 'dievaluasi', 'lagi', 'kebijakan', 'bebas', 'visa', 'terutama', 'untuk', 'negara', 'tiongkok', 'pak', '!', '!', 'bahaya', 'tersembunyi', 'martabat', 'RI']
['jangan', 'ngambang', 'aturan', 'logis', 'apa', 'undang', 'undang']
['Kebebasan', 'bersuara', 'berpendapat', 'memang', 'dijamin', 'UU', 'Tetapi', 'kebebasan', 'tersebut', 'tidak', 'harus', 'kebablasan', 'sehingga', 'menabrak', 'aturan', 'aturan', 'logis']


In [18]:
# 3.5 Simpan hasil tokenization
import os

os.makedirs('datapreprocessingcopy', exist_ok=True)

df.to_csv(
    'datapreprocessingcopy/data_after_tokenization.csv',
    index=False
)

print("data_after_tokenization.csv berhasil disimpan.")

data_after_tokenization.csv berhasil disimpan.


## Stemming

In [19]:
# 4.1 Install dan Import Sastrawi
import sys
import pandas as pd
import ast

# Cek apakah Sastrawi sudah terinstall
try:
    from Sastrawi.Stemmer.StemmerFactory import StemmerFactory
    print("Sastrawi sudah terinstall.")
except ImportError:
    print("Menginstall Sastrawi...")
    !{sys.executable} -m pip install Sastrawi
    from Sastrawi.Stemmer.StemmerFactory import StemmerFactory
    print("Sastrawi berhasil diinstall.")

Sastrawi sudah terinstall.


In [20]:
# 4.2 Load Data Tokenization
df = pd.read_csv('datapreprocessingcopy/data_after_tokenization.csv')

print("Dataset berhasil dimuat.")
print("Jumlah data:", len(df))

df.head()

Dataset berhasil dimuat.
Jumlah data: 13192


,no,timestamp,teks,teks_lengthening,teks_normalized,tokens
0,1,2016-12-30T06:37:56.000Z,ADIL loh utk yg punya kebijakan publik negara ...,ADIL loh utk yg punya kebijakan publik negara ...,ADIL loh untuk yang punya kebijakan publik neg...,"['ADIL', 'loh', 'untuk', 'yang', 'punya', 'keb..."
1,2,2016-12-30T06:30:36.000Z,Tertibkan Media Online DPR Pemerintah Jangan S...,Tertibkan Media Online DPR Pemerintah Jangan S...,Tertibkan Media Online DPR Pemerintah Jangan S...,"['Tertibkan', 'Media', 'Online', 'DPR', 'Pemer..."
2,3,2016-12-30T04:48:35.000Z,harus dievaluasi lg kebijakan bebas visa truta...,harus dievaluasi lg kebijakan bebas visa truta...,harus dievaluasi lagi kebijakan bebas visa ter...,"['harus', 'dievaluasi', 'lagi', 'kebijakan', '..."
3,4,2016-12-30T04:21:40.000Z,jangan ngambang aturan logis apa undang undang,jangan ngambang aturan logis apa undang undang,jangan ngambang aturan logis apa undang undang,"['jangan', 'ngambang', 'aturan', 'logis', 'apa..."
4,5,2016-12-30T02:36:13.000Z,Kebebasan bersuara berpendapat memang dijamin ...,Kebebasan bersuara berpendapat memang dijamin ...,Kebebasan bersuara berpendapat memang dijamin ...,"['Kebebasan', 'bersuara', 'berpendapat', 'mema..."


In [21]:
# 4.3 Convert string dan Handle NaN 
def convert_to_list(tokens):
    # Handle NaN / None
    if pd.isna(tokens):
        return []
    
    if isinstance(tokens, str):
        try:
            return ast.literal_eval(tokens)
        except:
            return []
    
    return tokens

df['tokens'] = df['tokens'].apply(convert_to_list)

# Validasi
print(f"Data dengan tokens kosong: {(df['tokens'].apply(len) == 0).sum()}")

Data dengan tokens kosong: 0


In [22]:
# 4.4 Inisialisasi Stemmer
factory = StemmerFactory()
stemmer = factory.create_stemmer()

In [23]:
# 4.5 Fungsi Stemming dengan Two-Layer Protection
def stem_tokens(tokens):
    if not isinstance(tokens, list):
        return []
   
    stemmed_tokens = []
   
    # === LAYER 1: VALID BASE WORDS (JANGAN DI-STEM) ===
    # Kata-kata yang SUDAH valid di KBBI/InSet, tidak perlu stemming
    valid_base_words = {
        # Function words (TIDAK PERNAH di-stem)
        'dengan', 'untuk', 'pada', 'dalam', 'ke', 'dari', 'di',
        'yang', 'ini', 'itu', 'ada', 'akan', 'sudah', 'telah',
        'tidak', 'bukan', 'jangan', 'belum', 'sangat', 'saja',
        'atau', 'dan', 'jika', 'kalau', 'karena', 'tetapi', 'namun',
        'saya', 'aku', 'kamu', 'anda', 'dia', 'mereka', 'kami', 'kita',
        
        # Kata politik/pemerintahan (SERING over-stemmed)
        'kebijakan', 'pemerintah', 'presiden', 'menteri', 'dewan',
        'rakyat', 'negara', 'bangsa', 'undang', 'hukum', 'rupiah',
        'anggaran', 'pembangunan', 'ekonomi', 'sosial', 'politik',
        
        # Kata sifat umum
        'baik', 'buruk', 'besar', 'kecil', 'baru', 'lama',
        'tinggi', 'rendah', 'jauh', 'dekat', 'cepat', 'lambat',
        'bijak', 'cerdas', 'pintar', 'kuat', 'lemah', 'kaya', 'miskin',
        
        # Kata kerja dasar
        'buat', 'makan', 'minum', 'jalan', 'lari', 'tidur',
        'kerja', 'belajar', 'main', 'lihat', 'dengar', 'bicara',
        'pikir', 'rasa', 'suka', 'benci', 'mau', 'bisa',
        
        # Kata benda umum
        'orang', 'hari', 'waktu', 'tempat', 'rumah', 'sekolah',
        'kerja', 'uang', 'mobil', 'motor', 'baju', 'makanan',
        'air', 'api', 'tanah', 'angin', 'bulan', 'matahari',
    }
   
    # === LAYER 2: STEM CORRECTION (FIX OVER-STEM) ===
    # Hanya untuk kata yang JELAS over-stemmed (bukan ambiguous)
    stem_correction = {
        'perintah': 'pemerintah',
        'milu': 'pemilu',
        'kesah': 'pengesahan',
        'anggar': 'anggaran',
        'selenggara': 'penyelenggaraan',
        'rakya': 'rakyat',
        'bangs': 'bangsa',
        'negar': 'negara',
        'tanggung': 'pertanggungjawaban',
        'wujud': 'perwujudan',
        'tahan': 'pertahanan',
        'jabat': 'jabatan',
        'duduk': 'kedudukan',
        'guna': 'penggunaan',
        'pakai': 'pemakaian',
        'olah': 'pengolahan',
        'tindak': 'tindakan',
        'laku': 'pelaksanaan',
        'atur': 'pengaturan',
        'pimpin': 'kepemimpinan',
        'rantang': 'perancangan',
        'rancang': 'rancangan',
        'putus': 'keputusan',
        'gugat': 'penggugatan',
        'tuntut': 'tuntutan',
        'desak': 'desakan',
        'dukung': 'dukungan',
        'tolak': 'penolakan',
        'setuju': 'persetujuan',
        'kritik': 'kritikan',
        'protes': 'protesan',
        'demo': 'demonstrasi',
        'aksi': 'gerakan',
        'massa': 'rakyat',
        'fraksi': 'partai',
        'koalisi': 'gabungan',
        'oposisi': 'penentang',
        'majoritas': 'mayoritas',
        'minoritas': 'kelompok',
        'berangan': 'bersebrangan',
    }
   
    for token in tokens:
        if token.isalpha():
            # 1. Cek apakah ALL CAPS
            is_all_caps = token.isupper() and len(token) > 1
            
            token_lower = token.lower()
            
            # 2. LAYER 1: Cek apakah sudah valid base word
            if token_lower in valid_base_words:
                # Kata sudah valid, TIDAK PERLU STEM!
                stemmed = token_lower
            else:
                # 3. LAYER 2: Stem dengan Sastrawi
                stemmed = stemmer.stem(token)
                
                # 4. Apply correction hanya untuk yang JELAS over-stemmed
                stemmed = stem_correction.get(stemmed, stemmed)
            
            # 5. Restore kapitalisasi jika ALL CAPS
            if is_all_caps:
                stemmed = stemmed.upper()
            # else: biarkan lowercase
            
            stemmed_tokens.append(stemmed)
        else:
            # Tanda baca, angka, dll - biarkan asli
            stemmed_tokens.append(token)
   
    return stemmed_tokens

In [24]:
# 4.6 Apply Stemming
df['tokens_stemmed'] = df['tokens'].apply(stem_tokens)

df[['tokens', 'tokens_stemmed']].head()

,tokens,tokens_stemmed
0,"[ADIL, loh, untuk, yang, punya, kebijakan, pub...","[ADIL, loh, untuk, yang, punya, kebijakan, pub..."
1,"[Tertibkan, Media, Online, DPR, Pemerintah, Ja...","[tertib, media, online, DPR, pemerintah, janga..."
2,"[harus, dievaluasi, lagi, kebijakan, bebas, vi...","[harus, evaluasi, lagi, kebijakan, bebas, visa..."
3,"[jangan, ngambang, aturan, logis, apa, undang,...","[jangan, ngambang, pengaturan, logis, apa, und..."
4,"[Kebebasan, bersuara, berpendapat, memang, dij...","[bebas, suara, dapat, memang, jamin, UU, tetap..."


In [25]:
# 4.7 Validasi Hasil
print("Contoh hasil stemming:\n")

for i in range(5):
    print("Original :", df['tokens'].iloc[i])
    print("Stemmed  :", df['tokens_stemmed'].iloc[i])
    print("-"*50)

Contoh hasil stemming:

Original : ['ADIL', 'loh', 'untuk', 'yang', 'punya', 'kebijakan', 'publik', 'negara', 'Ingat', 'yang', 'ini', '!', '!']
Stemmed  : ['ADIL', 'loh', 'untuk', 'yang', 'punya', 'kebijakan', 'publik', 'negara', 'ingat', 'yang', 'ini', '!', '!']
--------------------------------------------------
Original : ['Tertibkan', 'Media', 'Online', 'DPR', 'Pemerintah', 'Jangan', 'Sporadis', 'Apalagi', 'Selektif', 'Hanya', 'kepada', 'Media', 'yang', 'Berseberangan']
Stemmed  : ['tertib', 'media', 'online', 'DPR', 'pemerintah', 'jangan', 'sporadis', 'apalagi', 'selektif', 'hanya', 'kepada', 'media', 'yang', 'bersebrangan']
--------------------------------------------------
Original : ['harus', 'dievaluasi', 'lagi', 'kebijakan', 'bebas', 'visa', 'terutama', 'untuk', 'negara', 'tiongkok', 'pak', '!', '!', 'bahaya', 'tersembunyi', 'martabat', 'RI']
Stemmed  : ['harus', 'evaluasi', 'lagi', 'kebijakan', 'bebas', 'visa', 'utama', 'untuk', 'negara', 'tiongkok', 'pak', '!', '!', 'bahaya'

In [26]:
# 4.8 Menyimpan Hasil Stemming
import os

os.makedirs('datapreprocessingcopy', exist_ok=True)

df.to_csv(
    'datapreprocessingcopy/data_after_stemming.csv',
    index=False
)

print("data_after_stemming.csv berhasil disimpan.")

data_after_stemming.csv berhasil disimpan.


In [27]:
# Cell Diagnosa: Cek Kata yang Bermasalah
from collections import Counter

# Load data setelah stemming
df = pd.read_csv('datapreprocessingcopy/data_after_stemming.csv')
df['tokens'] = df['tokens'].apply(ast.literal_eval)
df['tokens_stemmed'] = df['tokens_stemmed'].apply(ast.literal_eval)

# Cari kata yang berubah
changed_words = []
for tokens, stemmed in zip(df['tokens'], df['tokens_stemmed']):
    for orig, stem in zip(tokens, stemmed):
        if orig.lower() != stem.lower() and orig.isalpha():
            changed_words.append((orig.lower(), stem.lower()))

# Hitung frekuensi
word_changes = Counter(changed_words)

print("TOP 50 KATA YANG BERUBAH SETELAH STEMMING:\n")
print(f"{'Kata Asli':<20} | {'Stem':<20} | {'Frekuensi'}")
print("-" * 60)
for (orig, stem), freq in word_changes.most_common(50):
    print(f"{orig:<20} | {stem:<20} | {freq}")

TOP 50 KATA YANG BERUBAH SETELAH STEMMING:

Kata Asli            | Stem                 | Frekuensi
------------------------------------------------------------
fraksi               | partai               | 737
pembahasan           | bahas                | 606
perwakilan           | wakil                | 586
menjadi              | jadi                 | 548
sebagai              | bagai                | 510
kunjungan            | kunjung              | 430
persidangan          | sidang               | 420
bersama              | sama                 | 371
kekerasan            | keras                | 359
meminta              | minta                | 344
terkait              | kait                 | 342
disahkan             | sah                  | 328
membahas             | bahas                | 292
terhadap             | hadap                | 291
tersebut             | sebut                | 234
melakukan            | pelaksanaan          | 234
penghapusan          | hapus           

# FORMATING DATA

In [31]:
# 5.1 Load data hasil stemming
import pandas as pd
import os

df = pd.read_csv('datapreprocessingcopy/data_after_stemming.csv')

print("Dataset berhasil dimuat.")
print("Jumlah data:", len(df))
print("Kolom:", df.columns.tolist())
df.head()

Dataset berhasil dimuat.
Jumlah data: 13192
Kolom: ['no', 'timestamp', 'teks', 'teks_lengthening', 'teks_normalized', 'tokens', 'tokens_stemmed']


,no,timestamp,teks,teks_lengthening,teks_normalized,tokens,tokens_stemmed
0,1,2016-12-30T06:37:56.000Z,ADIL loh utk yg punya kebijakan publik negara ...,ADIL loh utk yg punya kebijakan publik negara ...,ADIL loh untuk yang punya kebijakan publik neg...,"['ADIL', 'loh', 'untuk', 'yang', 'punya', 'keb...","['ADIL', 'loh', 'untuk', 'yang', 'punya', 'keb..."
1,2,2016-12-30T06:30:36.000Z,Tertibkan Media Online DPR Pemerintah Jangan S...,Tertibkan Media Online DPR Pemerintah Jangan S...,Tertibkan Media Online DPR Pemerintah Jangan S...,"['Tertibkan', 'Media', 'Online', 'DPR', 'Pemer...","['tertib', 'media', 'online', 'DPR', 'pemerint..."
2,3,2016-12-30T04:48:35.000Z,harus dievaluasi lg kebijakan bebas visa truta...,harus dievaluasi lg kebijakan bebas visa truta...,harus dievaluasi lagi kebijakan bebas visa ter...,"['harus', 'dievaluasi', 'lagi', 'kebijakan', '...","['harus', 'evaluasi', 'lagi', 'kebijakan', 'be..."
3,4,2016-12-30T04:21:40.000Z,jangan ngambang aturan logis apa undang undang,jangan ngambang aturan logis apa undang undang,jangan ngambang aturan logis apa undang undang,"['jangan', 'ngambang', 'aturan', 'logis', 'apa...","['jangan', 'ngambang', 'pengaturan', 'logis', ..."
4,5,2016-12-30T02:36:13.000Z,Kebebasan bersuara berpendapat memang dijamin ...,Kebebasan bersuara berpendapat memang dijamin ...,Kebebasan bersuara berpendapat memang dijamin ...,"['Kebebasan', 'bersuara', 'berpendapat', 'mema...","['bebas', 'suara', 'dapat', 'memang', 'jamin',..."


In [29]:
# 5.1.1 Load data hasil tokenisasi (ini digunakan sebagai perbandingan saja, namun hasilnya tidak disimpan)
import pandas as pd
import os
import ast 

df = pd.read_csv('datapreprocessingcopy/data_after_tokenization.csv')

print("Dataset berhasil dimuat.")
print("Jumlah data:", len(df))
print("Kolom:", df.columns.tolist())
df.head()

Dataset berhasil dimuat.
Jumlah data: 13192
Kolom: ['no', 'timestamp', 'teks', 'teks_lengthening', 'teks_normalized', 'tokens']


,no,timestamp,teks,teks_lengthening,teks_normalized,tokens
0,1,2016-12-30T06:37:56.000Z,ADIL loh utk yg punya kebijakan publik negara ...,ADIL loh utk yg punya kebijakan publik negara ...,ADIL loh untuk yang punya kebijakan publik neg...,"['ADIL', 'loh', 'untuk', 'yang', 'punya', 'keb..."
1,2,2016-12-30T06:30:36.000Z,Tertibkan Media Online DPR Pemerintah Jangan S...,Tertibkan Media Online DPR Pemerintah Jangan S...,Tertibkan Media Online DPR Pemerintah Jangan S...,"['Tertibkan', 'Media', 'Online', 'DPR', 'Pemer..."
2,3,2016-12-30T04:48:35.000Z,harus dievaluasi lg kebijakan bebas visa truta...,harus dievaluasi lg kebijakan bebas visa truta...,harus dievaluasi lagi kebijakan bebas visa ter...,"['harus', 'dievaluasi', 'lagi', 'kebijakan', '..."
3,4,2016-12-30T04:21:40.000Z,jangan ngambang aturan logis apa undang undang,jangan ngambang aturan logis apa undang undang,jangan ngambang aturan logis apa undang undang,"['jangan', 'ngambang', 'aturan', 'logis', 'apa..."
4,5,2016-12-30T02:36:13.000Z,Kebebasan bersuara berpendapat memang dijamin ...,Kebebasan bersuara berpendapat memang dijamin ...,Kebebasan bersuara berpendapat memang dijamin ...,"['Kebebasan', 'bersuara', 'berpendapat', 'mema..."


In [32]:
# 5.2 Pilih Kolom yang Dibutuhkan
df_final = df[['no', 'timestamp', 'teks', 'tokens_stemmed']].copy()

In [ ]:
# 5.2.1 Pilih Kolom yang Dibutuhkan (ini digunakan sebagai perbandingan saja, namun hasilnya tidak disimpan)
df_final = df[['no', 'timestamp', 'teks', 'tokens']].copy()

In [33]:
# 5.3 Convert tokens_stemmed ke teks
def safe_join(x):
    if isinstance(x, list):
        return ' '.join(x)
    
    try:
        x = ast.literal_eval(x)
        if isinstance(x, list):
            return ' '.join(x)
    except:
        pass
    
    return x


df_final['teks_processed'] = df_final['tokens_stemmed'].apply(safe_join)

In [ ]:
# 5.3.1 Convert tokens ke teks (ini digunakan sebagai perbandingan saja, namun hasilnya tidak disimpan)
def safe_join(x):
    if isinstance(x, list):
        return ' '.join(x)
   
    try:
        x = ast.literal_eval(x)
        if isinstance(x, list):
            return ' '.join(x)
    except:
        pass
   
    return x

df_final['teks_processed'] = df_final['tokens'].apply(safe_join)

In [34]:
# 5.4 Hapus kolom lama (biar tidak double)
df_final = df_final.drop(columns=['tokens_stemmed'])

In [ ]:
# 5.4.1 Hapus kolom lama (biar tidak double) (ini digunakan sebagai perbandingan saja, namun hasilnya tidak disimpan)
df_final = df_final.drop(columns=['tokens'])

In [35]:
# 5.5 Validasi
print("\n=== Validasi Data Final ===")
print(f"Jumlah data: {len(df_final)}")
print(f"Kolom: {df_final.columns.tolist()}")
print(f"Data kosong: {df_final['teks_processed'].isna().sum()}")

# Cek sampel
print("\n=== Sampel Data ===")
print(df_final[['teks', 'teks_processed']].head(3))


=== Validasi Data Final ===
Jumlah data: 13192
Kolom: ['no', 'timestamp', 'teks', 'teks_processed']
Data kosong: 0

=== Sampel Data ===
                                                teks  \
0  ADIL loh utk yg punya kebijakan publik negara ...   
1  Tertibkan Media Online DPR Pemerintah Jangan S...   
2  harus dievaluasi lg kebijakan bebas visa truta...   

                                      teks_processed  
0  ADIL loh untuk yang punya kebijakan publik neg...  
1  tertib media online DPR pemerintah jangan spor...  
2  harus evaluasi lagi kebijakan bebas visa utama...  


In [ ]:
# 5.5.1 Validasi (ini digunakan sebagai perbandingan saja, namun hasilnya tidak disimpan)
print("\n=== Validasi Data Final ===")
print(f"Jumlah data: {len(df_final)}")
print(f"Kolom: {df_final.columns.tolist()}")
print(f"Data kosong: {df_final['teks_processed'].isna().sum()}")

# Cek sampel
print("\n=== Sampel Data ===")
print(df_final[['teks', 'teks_processed']].head(3))


=== Validasi Data Final ===
Jumlah data: 13243
Kolom: ['no', 'timestamp', 'teks', 'teks_processed']
Data kosong: 0

=== Sampel Data ===
                                                teks  \
0  ADIL loh utk yg punya kebijakan publik negara ...   
1  Tertibkan Media Online DPR Pemerintah Jangan S...   
2  harus dievaluasi lg kebijakan bebas visa truta...   

                                      teks_processed  
0  ADIL loh untuk yang punya kebijakan publik neg...  
1  Tertibkan Media Online DPR Pemerintah Jangan S...  
2  harus dievaluasi lagi kebijakan bebas visa ter...  


In [23]:
# 5.6 Simpan sebagai data preprocessing final
os.makedirs('datapreprocessingcopy', exist_ok=True)

df_final.to_csv(
    '../datapreprocessingcopy/data_preprocessing_final.csv',
    index=False,
    encoding='utf-8'
)

print("\n data_preprocessing_final.csv berhasil disimpan!")
print(f"Lokasi: ../datapreprocessingcopy/data_preprocessing_final.csv")

NameError: name 'df_final' is not defined

In [ ]:
# 5.6.1 Simpan sebagai data preprocessing final (ini digunakan sebagai perbandingan saja, namun hasilnya tidak disimpan)
os.makedirs('datapreprocessingcopy', exist_ok=True)

df_final.to_csv(
    'datapreprocessingcopy/data_preprocessingcopy_final.csv',
    index=False,
    encoding='utf-8'
)

print("\n data_preprocessingcopy_final.csv berhasil disimpan!")
print(f"Lokasi: datapreprocessingcopy/data_preprocessingcopy_final.csv")

NameError: name 'df_final' is not defined